# Databricks ReAct Inference System

This notebook provides a high-level interface for Databricks ReAct Inference that uses your existing `predict()` function instead of VLLM servers.

**Key Features:**
- ✅ Uses properly migrated `DatabricksMultiTurnReactAgent`
- ✅ Integrates with `databricks_multi_react` for batch processing
- ✅ Full research capabilities with tools (search, scholar, python, file parsing)
- ✅ Environment management with dotenv
- ✅ Compatible with your existing MLflow setup

**Based on:** `inference/databricks_react_inference.py`

## 1. Setup and Imports

In [ ]:
import os
import sys
import json
from typing import List, Dict, Optional, Any, Callable
import tempfile

# Add inference directory to path
sys.path.append('inference')

from dotenv import load_dotenv

# Import the properly migrated components
from databricks_react_agent import DatabricksMultiTurnReactAgent
from databricks_multi_react import run_databricks_multi_react

print("✅ All imports successful!")
print("✅ Ready to use Databricks ReAct Inference System")

## 2. Environment Configuration

Load environment variables from `.env` file for API keys and configuration.

In [ ]:
# Load environment variables
load_dotenv('inference/.env')

# Display current environment configuration
print("🔧 Environment Configuration:")
print(f"   Temperature: {os.environ.get('TEMPERATURE', '0.85')}")
print(f"   Presence Penalty: {os.environ.get('PRESENCE_PENALTY', '1.1')}")
print(f"   Rollout Count: {os.environ.get('ROLLOUT_COUNT', '3')}")
print(f"   Max Workers: {os.environ.get('MAX_WORKERS', '30')}")

# Check API key status
api_keys = ['SERPER_KEY_ID', 'JINA_API_KEYS', 'DASHSCOPE_API_KEY']
print("\n🔑 API Keys Status:")
for key in api_keys:
    status = "✅ Configured" if os.environ.get(key) and os.environ[key] != f'your_{key.lower()}' else "⚠️ Not configured"
    print(f"   {key}: {status}")

print("\n📝 To configure API keys:")
print("   1. Copy inference/.env.example to inference/.env")
print("   2. Fill in your actual API keys")
print("   3. Restart this notebook")

## 3. DatabricksReactInference Class Definition

In [ ]:
class DatabricksReactInference:
    """
    High-level interface for Databricks ReAct Inference
    Uses the properly migrated DatabricksMultiTurnReactAgent
    """

    def __init__(self, predict_function: Callable[[str], str]):
        """
        Initialize with your predict function

        Args:
            predict_function: Your existing predict() function that takes a string and returns inference
        """
        self.predict_function = predict_function

        # Load environment variables
        load_dotenv('inference/.env')

        print("✅ DatabricksReactInference initialized with predict() function")
        print("   Using properly migrated DatabricksMultiTurnReactAgent and databricks_multi_react")

    def setup_environment(self,
                         dataset: str = "research_dataset",
                         output_path: str = "/tmp/databricks_output",
                         temperature: float = None,
                         presence_penalty: float = None,
                         rollout_count: int = None,
                         max_workers: int = None,
                         # API Keys
                         serper_key: Optional[str] = None,
                         jina_api_keys: Optional[str] = None,
                         api_key: Optional[str] = None,
                         api_base: Optional[str] = None,
                         summary_model_name: Optional[str] = None,
                         dashscope_api_key: Optional[str] = None,
                         dashscope_api_base: Optional[str] = None,
                         **kwargs) -> Dict[str, str]:
        """
        Set up environment variables for inference
        """

        # Use provided values or defaults from environment
        env_vars = {
            'DATASET': dataset,
            'OUTPUT_PATH': output_path,
            'TEMPERATURE': str(temperature or os.environ.get('TEMPERATURE', 0.85)),
            'PRESENCE_PENALTY': str(presence_penalty or os.environ.get('PRESENCE_PENALTY', 1.1)),
            'ROLLOUT_COUNT': str(rollout_count or os.environ.get('ROLLOUT_COUNT', 3)),
            'MAX_WORKERS': str(max_workers or os.environ.get('MAX_WORKERS', 30))
        }

        # API Keys (only set if provided)
        if serper_key or os.environ.get('SERPER_KEY_ID'):
            env_vars['SERPER_KEY_ID'] = serper_key or os.environ.get('SERPER_KEY_ID')
        if jina_api_keys or os.environ.get('JINA_API_KEYS'):
            env_vars['JINA_API_KEYS'] = jina_api_keys or os.environ.get('JINA_API_KEYS')
        if api_key or os.environ.get('API_KEY'):
            env_vars['API_KEY'] = api_key or os.environ.get('API_KEY')
        if api_base or os.environ.get('API_BASE'):
            env_vars['API_BASE'] = api_base or os.environ.get('API_BASE')
        if summary_model_name or os.environ.get('SUMMARY_MODEL_NAME'):
            env_vars['SUMMARY_MODEL_NAME'] = summary_model_name or os.environ.get('SUMMARY_MODEL_NAME')
        if dashscope_api_key or os.environ.get('DASHSCOPE_API_KEY'):
            env_vars['DASHSCOPE_API_KEY'] = dashscope_api_key or os.environ.get('DASHSCOPE_API_KEY')
        if dashscope_api_base or os.environ.get('DASHSCOPE_API_BASE'):
            env_vars['DASHSCOPE_API_BASE'] = dashscope_api_base or os.environ.get('DASHSCOPE_API_BASE')

        # Set environment variables
        for key, value in env_vars.items():
            os.environ[key] = str(value)

        print("✅ Environment configured:")
        print(f"   Temperature: {env_vars['TEMPERATURE']}")
        print(f"   Presence Penalty: {env_vars['PRESENCE_PENALTY']}")
        print(f"   Rollout Count: {env_vars['ROLLOUT_COUNT']}")
        print(f"   Max Workers: {env_vars['MAX_WORKERS']}")

        # Print API key status
        api_keys = ['SERPER_KEY_ID', 'JINA_API_KEYS', 'DASHSCOPE_API_KEY']
        for key in api_keys:
            if key in env_vars:
                print(f"   {key}: ✅ Configured")
            else:
                print(f"   {key}: ⚠️ Not configured")

        return env_vars

    def research_question(self,
                         question: str,
                         model_name: str = "tongyi-deep-research") -> Dict[str, Any]:
        """
        Process a single research question using DatabricksMultiTurnReactAgent

        Args:
            question: Research question to process
            model_name: Model identifier

        Returns:
            Result dictionary with prediction and metadata
        """
        print(f"🔬 Processing research question: {question}")

        # Initialize agent
        llm_cfg = {
            'model': model_name,
            'generate_cfg': {
                'max_input_tokens': 320000,
                'max_retries': 10,
                'temperature': float(os.environ.get('TEMPERATURE', 0.85)),
                'top_p': 0.95,
                'presence_penalty': float(os.environ.get('PRESENCE_PENALTY', 1.1))
            },
            'model_type': 'databricks'
        }

        agent = DatabricksMultiTurnReactAgent(
            predict_function=self.predict_function,
            llm=llm_cfg,
            function_list=["search", "visit", "google_scholar", "PythonInterpreter"]
        )

        # Process question
        data = {
            'item': {
                'question': question,
                'answer': ''  # Reference answer (empty for new questions)
            }
        }

        result = agent._run(data, model_name)
        return result

    def batch_research(self,
                      questions: List[str],
                      dataset: str = "research_dataset",
                      output_path: str = "/tmp/databricks_output",
                      model_name: str = "tongyi-deep-research",
                      rollout_count: int = None,
                      max_workers: int = None,
                      **kwargs) -> bool:
        """
        Process multiple research questions using the complete databricks_multi_react system

        Args:
            questions: List of research questions
            dataset: Dataset name
            output_path: Output directory
            model_name: Model identifier
            rollout_count: Number of rollouts per question
            max_workers: Maximum worker threads
            **kwargs: Additional parameters

        Returns:
            True if successful, False otherwise
        """
        print(f"📚 Processing {len(questions)} research questions...")

        # Create temporary dataset file
        with tempfile.NamedTemporaryFile(mode='w', suffix='.jsonl', delete=False) as f:
            for question in questions:
                item = {
                    'question': question,
                    'answer': ''  # Empty reference answer
                }
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
            temp_data_file = f.name

        try:
            # Run complete system
            success = run_databricks_multi_react(
                predict_function=self.predict_function,
                dataset=dataset,
                output_path=output_path,
                model_name=model_name,
                temperature=float(os.environ.get('TEMPERATURE', 0.85)),
                presence_penalty=float(os.environ.get('PRESENCE_PENALTY', 1.1)),
                rollout_count=rollout_count or int(os.environ.get('ROLLOUT_COUNT', 3)),
                max_workers=max_workers or int(os.environ.get('MAX_WORKERS', 30)),
                data_file=temp_data_file,
                **kwargs
            )

            return success

        finally:
            # Clean up temporary file
            try:
                os.unlink(temp_data_file)
            except OSError:
                pass

    def run_complete_inference(self,
                              questions: Optional[List[str]] = None,
                              dataset: str = "research_dataset",
                              output_path: str = "/tmp/databricks_output",
                              model_name: str = "tongyi-deep-research",
                              data_file: Optional[str] = None,
                              **kwargs) -> Dict[str, Any]:
        """
        Run complete ReAct inference pipeline - equivalent to run_react_infer.sh

        Args:
            questions: List of questions to process (optional if data_file provided)
            dataset: Dataset name
            output_path: Output directory
            model_name: Model identifier
            data_file: Path to existing data file
            **kwargs: Additional parameters

        Returns:
            Dictionary with results and status information
        """
        print("🚀 Starting Complete Databricks ReAct Inference Pipeline")
        print("=" * 60)

        result = {
            "status": "started",
            "dataset": dataset,
            "output_path": output_path,
            "model_name": model_name,
            "questions_processed": 0,
            "success": False,
            "error": None
        }

        try:
            # Setup environment
            self.setup_environment(
                dataset=dataset,
                output_path=output_path,
                **kwargs
            )

            # Use provided questions or data file
            if questions and not data_file:
                success = self.batch_research(
                    questions=questions,
                    dataset=dataset,
                    output_path=output_path,
                    model_name=model_name,
                    **kwargs
                )
                result["questions_processed"] = len(questions)
            else:
                # Use run_databricks_multi_react directly
                success = run_databricks_multi_react(
                    predict_function=self.predict_function,
                    dataset=dataset,
                    output_path=output_path,
                    model_name=model_name,
                    data_file=data_file,
                    **kwargs
                )
                result["questions_processed"] = "See output files"

            if success:
                result["status"] = "completed"
                result["success"] = True
                print("✅ Complete ReAct inference pipeline completed successfully!")
            else:
                result["status"] = "failed"
                result["error"] = "Pipeline execution failed"
                print("❌ Pipeline execution failed")

        except Exception as e:
            print(f"❌ Error during inference: {e}")
            result["status"] = "failed"
            result["error"] = str(e)

        return result

print("✅ DatabricksReactInference class defined successfully!")

## 4. Generic Predict Function Setup

You can use **any** predict function that takes a string prompt and returns a string response.

In [ ]:
# Option 1: Use your own predict function directly
def your_predict_function(prompt: str) -> str:
    """
    Replace this with your actual predict function
    
    Examples:
    - Databricks model serving endpoint
    - HuggingFace transformers
    - OpenAI API
    - Local model inference
    - Any custom model wrapper
    
    Args:
        prompt: Input text prompt
        
    Returns:
        Generated text response
    """
    # Example implementations:
    
    # For Databricks model serving:
    # response = requests.post(your_databricks_endpoint, json={"inputs": prompt})
    # return response.json()["predictions"][0]
    
    # For HuggingFace transformers:
    # inputs = tokenizer.encode(prompt, return_tensors="pt")
    # outputs = model.generate(inputs, max_length=2048, temperature=0.85)
    # return tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # For OpenAI API:
    # response = openai.ChatCompletion.create(messages=[{"role": "user", "content": prompt}])
    # return response.choices[0].message.content
    
    # Placeholder for demonstration:
    return f"Generated response for: {prompt[:100]}..."

# Option 2: MLflow wrapper (if you're using MLflow)
def create_mlflow_predict_wrapper(logger, context=None):
    """
    Create a predict function wrapper for MLflow models
    
    Args:
        logger: Your MLflow model logger/wrapper
        context: MLflow context (optional)
        
    Returns:
        Function that takes a string and returns prediction
    """
    def predict_function(prompt: str) -> str:
        model_input = [{
            "prompt": prompt,
            "max_length": 2048,
            "temperature": float(os.environ.get('TEMPERATURE', 0.85)),
            "presence_penalty": float(os.environ.get('PRESENCE_PENALTY', 1.1))
        }]
        
        try:
            results = logger.predict(context, model_input)
            if isinstance(results, list) and len(results) > 0:
                return str(results[0])
            else:
                return str(results)
        except Exception as e:
            return f"Error in prediction: {e}"
    
    return predict_function

# Option 3: Databricks serving endpoint
def create_databricks_predict_function(endpoint_url: str, token: str):
    """
    Create predict function for Databricks model serving endpoint
    
    Args:
        endpoint_url: Your Databricks model serving endpoint
        token: Databricks access token
        
    Returns:
        Function that takes a string and returns prediction
    """
    import requests
    
    def predict_function(prompt: str) -> str:
        headers = {"Authorization": f"Bearer {token}"}
        data = {
            "inputs": [prompt],
            "params": {
                "temperature": float(os.environ.get('TEMPERATURE', 0.85)),
                "max_tokens": 2048
            }
        }
        
        try:
            response = requests.post(endpoint_url, headers=headers, json=data)
            response.raise_for_status()
            result = response.json()
            return result["predictions"][0]
        except Exception as e:
            return f"Error in prediction: {e}"
    
    return predict_function

print("✅ Generic predict function templates defined!")
print("\n🎯 Choose your option:")
print("   1. Modify 'your_predict_function()' with your implementation")
print("   2. Use 'create_mlflow_predict_wrapper()' for MLflow models") 
print("   3. Use 'create_databricks_predict_function()' for serving endpoints")
print("   4. Create your own custom wrapper function")

## 5. Initialize Your Predict Function

Choose and set up your predict function based on your setup.

In [ ]:
# =============================================================================
# CHOOSE YOUR PREDICT FUNCTION SETUP
# =============================================================================

# Option A: Use your own custom predict function
print("🎯 Setting up predict function...")

# Method 1: Direct function (modify this with your implementation)
def my_predict_function(prompt: str) -> str:
    """
    Replace this with YOUR actual predict function
    
    This should call your Databricks model, HuggingFace model, 
    OpenAI API, or any other inference endpoint/method.
    """
    
    # REPLACE THIS SECTION WITH YOUR ACTUAL IMPLEMENTATION:
    # ====================================================
    
    # Example for Databricks model serving:
    # import requests
    # response = requests.post(
    #     "your-databricks-endpoint-url",
    #     headers={"Authorization": f"Bearer {your_token}"},
    #     json={"inputs": [prompt]}
    # )
    # return response.json()["predictions"][0]
    
    # Example for HuggingFace transformers:
    # from transformers import pipeline
    # generator = pipeline("text-generation", model="your-model")
    # result = generator(prompt, max_length=2048, temperature=0.85)
    # return result[0]["generated_text"]
    
    # Placeholder response for demo:
    return f"Demo response to prompt: {prompt[:100]}..."

# Method 2: MLflow wrapper (uncomment if using MLflow)
# try:
#     sys.path.append('..')
#     from huggingface_mlflow_logger import HuggingFaceMLflowLogger
#     
#     logger = HuggingFaceMLflowLogger(
#         model_name="your-model-name",
#         hf_token="your-hf-token"
#     )
#     
#     my_predict_function = create_mlflow_predict_wrapper(logger)
#     print("✅ Using MLflow predict function")
# 
# except ImportError:
#     print("⚠️ MLflow not available, using direct function")

# Method 3: Databricks serving endpoint (uncomment if using serving endpoint)
# my_predict_function = create_databricks_predict_function(
#     endpoint_url="https://your-workspace.databricks.com/serving-endpoints/your-endpoint/invocations",
#     token="your-databricks-token"
# )

# =============================================================================
# VERIFY YOUR PREDICT FUNCTION
# =============================================================================

print("\n🧪 Testing predict function...")
test_prompt = "Hello, this is a test prompt."
try:
    test_response = my_predict_function(test_prompt)
    print(f"✅ Predict function working!")
    print(f"   Input: {test_prompt}")
    print(f"   Output: {test_response[:100]}...")
    
    # Use this as your predict function
    predict_func = my_predict_function
    
except Exception as e:
    print(f"❌ Predict function failed: {e}")
    print("   Please check your implementation above")
    
    # Fallback to demo function
    def demo_predict_function(prompt: str) -> str:
        return f"Demo response for: {prompt[:50]}..."
    
    predict_func = demo_predict_function
    print("   Using demo function for now")

print("\n✅ Predict function ready for ReAct system!")

## 6. Initialize the ReAct Inference System

In [ ]:
# Initialize the Databricks ReAct Inference system
inference_system = DatabricksReactInference(predict_func)

print("\n🎯 ReAct Inference System Ready!")
print("   Available methods:")
print("   • research_question() - Process single question")
print("   • batch_research() - Process multiple questions")
print("   • run_complete_inference() - Full pipeline")

## 7. Example 1: Single Research Question

Process a single research question using the ReAct agent with tools.

In [ ]:
# Example research question
research_question = "What are the key advantages of attention mechanisms in transformer models?"

print(f"🔬 Research Question: {research_question}")
print("\n🔄 Starting ReAct workflow...")
print("   This will:")
print("   1. Use your predict() function to reason about the question")
print("   2. Decide what tools to use (search, scholar, python, etc.)")
print("   3. Execute tools and gather information")
print("   4. Use your predict() function again to synthesize results")
print("   5. Repeat until final answer is reached")

try:
    # Process the research question
    result = inference_system.research_question(research_question)
    
    print("\n✅ Research completed!")
    print(f"   Status: {result.get('termination', 'Unknown')}")
    print(f"   Answer: {result.get('prediction', 'No answer found')}")
    print(f"   Total rounds: {len(result.get('messages', [])) // 2}")
    
except Exception as e:
    print(f"❌ Error during research: {e}")
    print("   This might be due to missing API keys or import issues.")
    print("   Check your .env file and ensure all tools are accessible.")

## 8. Example 2: Batch Research Questions

Process multiple research questions using the complete batch system.

In [ ]:
# Example batch of research questions
research_questions = [
    "How do attention mechanisms work in neural networks?",
    "What are the differences between GPT and BERT architectures?",
    "How can transformer models be made more efficient?"
]

print(f"📚 Batch Research: {len(research_questions)} questions")
for i, q in enumerate(research_questions, 1):
    print(f"   {i}. {q}")

print("\n🚀 Starting batch processing...")
print("   This uses the complete databricks_multi_react system with:")
print("   • Threading for parallel processing")
print("   • Multiple rollouts per question")
print("   • Full ReAct workflow for each question")
print("   • Results saved to output files")

try:
    # Process batch of questions
    success = inference_system.batch_research(
        questions=research_questions,
        dataset="demo_research",
        output_path="/tmp/react_batch_output",
        rollout_count=2,  # Reduced for demo
        max_workers=2     # Reduced for demo
    )
    
    if success:
        print("\n✅ Batch research completed successfully!")
        print("   Results saved to: /tmp/react_batch_output/")
        print("   Check the output directory for detailed results.")
    else:
        print("\n❌ Batch research failed")
        
except Exception as e:
    print(f"❌ Error during batch research: {e}")
    print("   This might be due to missing dependencies or configuration issues.")

## 9. Example 3: Complete Inference Pipeline

Run the complete pipeline equivalent to `run_react_infer.sh`.

In [ ]:
# Complete inference pipeline
pipeline_questions = [
    "What are the latest developments in large language models?",
    "How can we improve the efficiency of transformer models?"
]

print("🚀 Complete Inference Pipeline Demo")
print("   Equivalent to running run_react_infer.sh")
print(f"   Processing {len(pipeline_questions)} questions")

try:
    # Run complete pipeline
    result = inference_system.run_complete_inference(
        questions=pipeline_questions,
        dataset="complete_demo",
        output_path="/tmp/react_complete_output",
        model_name="tongyi-deep-research",
        temperature=0.85,
        presence_penalty=1.1,
        rollout_count=1,  # Reduced for demo
        max_workers=2     # Reduced for demo
    )
    
    print(f"\n📊 Pipeline Results:")
    print(f"   Status: {result['status']}")
    print(f"   Dataset: {result['dataset']}")
    print(f"   Output Path: {result['output_path']}")
    print(f"   Questions Processed: {result['questions_processed']}")
    print(f"   Success: {result['success']}")
    
    if result['error']:
        print(f"   Error: {result['error']}")
        
except Exception as e:
    print(f"❌ Error during complete pipeline: {e}")

## 10. Configuration and API Keys

Information about setting up the system properly.

In [ ]:
print("⚙️ CONFIGURATION GUIDE")
print("=" * 50)

print("\n📁 File Structure:")
print("   inference/.env.example → Copy to inference/.env")
print("   inference/databricks_react_agent.py → Modified agent")
print("   inference/databricks_multi_react.py → Batch processor")
print("   inference/tool_*.py → All research tools")

print("\n🔑 Required API Keys (Optional but Recommended):")
api_info = {
    'SERPER_KEY_ID': {
        'purpose': 'Web search functionality',
        'url': 'https://serper.dev/',
        'tools': 'Search tool, Google Scholar'
    },
    'JINA_API_KEYS': {
        'purpose': 'Webpage content reading',
        'url': 'https://jina.ai/',
        'tools': 'Visit tool (webpage analysis)'
    },
    'DASHSCOPE_API_KEY': {
        'purpose': 'Advanced file parsing, video/audio analysis',
        'url': 'https://dashscope.aliyun.com/',
        'tools': 'File parsing tool, video analysis'
    }
}

for key, info in api_info.items():
    configured = os.environ.get(key) and os.environ[key] != f'your_{key.lower()}':
    status = "✅ Configured" if configured else "⚠️ Not configured"
    print(f"\n   {key}: {status}")
    print(f"      Purpose: {info['purpose']}")
    print(f"      Get key at: {info['url']}")
    print(f"      Enables: {info['tools']}")

print("\n🔧 Model Parameters:")
params = {
    'TEMPERATURE': os.environ.get('TEMPERATURE', '0.85'),
    'PRESENCE_PENALTY': os.environ.get('PRESENCE_PENALTY', '1.1'),
    'ROLLOUT_COUNT': os.environ.get('ROLLOUT_COUNT', '3'),
    'MAX_WORKERS': os.environ.get('MAX_WORKERS', '30')
}

for param, value in params.items():
    print(f"   {param}: {value}")

print("\n💡 Tips:")
print("   • The system works without API keys (limited functionality)")
print("   • Add API keys gradually as you need more capabilities")
print("   • Start with SERPER_KEY for web search - most useful")
print("   • DASHSCOPE_API_KEY adds advanced file analysis capabilities")
print("   • All tools fall back gracefully when keys are missing")

## 11. Summary and Next Steps

In [ ]:
print("🎯 GENERIC PREDICT FUNCTION REACT SYSTEM READY!")
print("=" * 60)

print("\n✅ What You Have Now:")
print("   • Complete ReAct system using ANY predict() function")
print("   • Works with Databricks, HuggingFace, OpenAI, or custom models")
print("   • All research tools: search, scholar, python, file parsing")
print("   • Batch processing with threading and rollouts")
print("   • Same output format as original run_multi_react.py")
print("   • Environment management with dotenv")

print("\n🔧 Predict Function Options:")
print("   1. Databricks model serving endpoint")
print("   2. HuggingFace transformers pipeline")
print("   3. OpenAI API calls")
print("   4. MLflow model wrapper")
print("   5. Any custom inference function")
print("   6. Local model inference")

print("\n🚀 How to Use:")
print("   1. Single Question:")
print("      result = inference_system.research_question('Your question')")
print("   2. Batch Questions:")
print("      success = inference_system.batch_research(['Q1', 'Q2'])")
print("   3. Complete Pipeline:")
print("      result = inference_system.run_complete_inference(questions=...)")

print("\n📋 Setup Steps:")
print("   1. Replace 'my_predict_function()' with your actual implementation")
print("   2. Test your predict function")
print("   3. Configure API keys in inference/.env (optional)")
print("   4. Run research questions!")

print("\n📁 Key Files:")
print("   • This notebook: High-level interface")
print("   • inference/databricks_react_agent.py: Core agent logic")
print("   • inference/databricks_multi_react.py: Batch processing")
print("   • inference/.env: Configuration (create from .env.example)")

print("\n🎉 Ready for Any Model!")
print("   Your predict() function + ReAct tools = Complete research system")